# ORCA — Drift & Origin Reconstruction Experiments

## Objective

Estimate the probable origin of an observed oil spill using
ocean currents, wind forcing, and stochastic Lagrangian particle
advection.

### Core pipeline

Observed Spill
      ↓
Spill Centroid
      ↓
Environmental Forcing
      ↓
Backward Particle Advection
      ↓
Diffusion / Uncertainty
      ↓
Origin Probability Field
      ↓
Probable Origin Zone

## Experiment types

1. Deterministic forward advection
2. Deterministic backward hindcast
3. Stochastic backward particle simulation
4. Origin probability estimation
5. Uncertainty-region estimation

### Current limitation

This notebook initially uses synthetic environmental conditions.
Real oceanographic and meteorological datasets will be connected later.

In [ ]:
import math
import random
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Random seed:", SEED)

In [ ]:
# =========================
# ORCA Drift Configuration
# =========================

LOOKBACK_HOURS = 24
FORECAST_HOURS = 24

TIME_STEP_HOURS = 1

NUM_PARTICLES = 2000

# Oil windage coefficient
WINDAGE = 0.03

# Horizontal diffusion coefficient
# m²/s
DIFFUSION_COEFFICIENT = 5.0

# Observed spill location
OBSERVED_LAT = 13.100
OBSERVED_LON = 77.100

print("Lookback:", LOOKBACK_HOURS, "hours")
print("Particles:", NUM_PARTICLES)
print("Windage:", WINDAGE)
print("Diffusion coefficient:", DIFFUSION_COEFFICIENT)

In [ ]:
@dataclass
class Environment:
    current_u_mps: float
    current_v_mps: float

    wind_u_mps: float
    wind_v_mps: float

    diffusion_m2s: float

In [ ]:
environment = Environment(
    current_u_mps=0.25,
    current_v_mps=0.10,

    wind_u_mps=4.0,
    wind_v_mps=1.5,

    diffusion_m2s=DIFFUSION_COEFFICIENT,
)

environment

In [ ]:
@dataclass
class Particle:
    latitude: float
    longitude: float

particle = Particle(
    latitude=OBSERVED_LAT,
    longitude=OBSERVED_LON
)

particle

In [ ]:
EARTH_RADIUS_M = 6_371_000.0


def meters_to_latitude(meters):
    return meters / 111_320.0


def meters_to_longitude(meters, latitude):
    cos_lat = max(
        0.1,
        math.cos(math.radians(latitude))
    )

    return meters / (111_320.0 * cos_lat)

In [ ]:
def advect_particle(
    particle,
    hours,
    environment,
    windage=0.03
):
    """
    Deterministic particle advection.

    Positive hours:
        forward simulation

    Negative hours:
        backward simulation
    """

    total_u = (
        environment.current_u_mps
        + windage * environment.wind_u_mps
    )

    total_v = (
        environment.current_v_mps
        + windage * environment.wind_v_mps
    )

    seconds = hours * 3600.0

    dx = total_u * seconds
    dy = total_v * seconds

    lat_delta = meters_to_latitude(dy)

    lon_delta = meters_to_longitude(
        dx,
        particle.latitude
    )

    return Particle(
        latitude=particle.latitude + lat_delta,
        longitude=particle.longitude + lon_delta
    )

In [ ]:
def simulate_forward(
    initial_particle,
    hours,
    time_step_hours,
    environment,
    windage=0.03
):

    trajectory = []

    particle = initial_particle

    num_steps = int(hours / time_step_hours)

    for step in range(num_steps + 1):

        trajectory.append({
            "hour": step * time_step_hours,
            "latitude": particle.latitude,
            "longitude": particle.longitude
        })

        particle = advect_particle(
            particle,
            time_step_hours,
            environment,
            windage
        )

    return pd.DataFrame(trajectory)

In [ ]:
forward_df = simulate_forward(
    particle,
    FORECAST_HOURS,
    TIME_STEP_HOURS,
    environment,
    WINDAGE
)

forward_df.head()

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(
    forward_df["longitude"],
    forward_df["latitude"]
)

plt.scatter(
    OBSERVED_LON,
    OBSERVED_LAT,
    s=80,
    label="Initial position"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Deterministic Forward Oil-Slick Trajectory")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
def simulate_backward(
    observed_particle,
    hours,
    time_step_hours,
    environment,
    windage=0.03
):

    trajectory = []

    particle = observed_particle

    num_steps = int(hours / time_step_hours)

    for step in range(num_steps + 1):

        trajectory.append({
            "hours_before_observation": step * time_step_hours,
            "latitude": particle.latitude,
            "longitude": particle.longitude
        })

        particle = advect_particle(
            particle,
            -time_step_hours,
            environment,
            windage
        )

    return pd.DataFrame(trajectory)

In [ ]:
backward_df = simulate_backward(
    particle,
    LOOKBACK_HOURS,
    TIME_STEP_HOURS,
    environment,
    WINDAGE
)

backward_df.head()

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(
    backward_df["longitude"],
    backward_df["latitude"],
    marker="o",
    markersize=3
)

plt.scatter(
    OBSERVED_LON,
    OBSERVED_LAT,
    s=100,
    label="Observed spill"
)

origin = backward_df.iloc[-1]

plt.scatter(
    origin["longitude"],
    origin["latitude"],
    s=100,
    label="Estimated origin"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Deterministic Backward Hindcast")
plt.legend()
plt.grid(True)

plt.show()

# Stochastic Lagrangian Oil-Particle Model

The core Lagrangian stochastic model governing oil-particle motion is:

$$
\frac{d\mathbf{x}}{dt}
=
\mathbf{u}_c(\mathbf{x},t)
+
\alpha \mathbf{u}_w(\mathbf{x},t)
+
\sqrt{2K}\,\boldsymbol{\xi}(t)
$$

where:

- $\mathbf{x}(t) = [x(t),y(t)]$ is the oil-particle position.
- $\mathbf{u}_c(\mathbf{x},t)$ is the ocean surface current velocity.
- $\mathbf{u}_w(\mathbf{x},t)$ is the wind velocity.
- $\alpha$ is the windage coefficient.
- $K$ is the horizontal turbulent diffusivity.
- $\boldsymbol{\xi}(t)$ represents stochastic forcing.

This is a stochastic differential equation (SDE) representing an
advection-diffusion process.

The deterministic component represents bulk transport by currents and
wind, while the stochastic component represents unresolved turbulent
mixing.

Because of the stochastic term, an individual particle trajectory is
not interpreted as the exact oil trajectory. Instead, an ensemble of
particles is simulated and interpreted statistically.

## Discrete Numerical Form

For a timestep $\Delta t$:

$$
\mathbf{x}_{t+\Delta t}
=
\mathbf{x}_t
+
\left[
\mathbf{u}_c(\mathbf{x}_t,t)
+
\alpha\mathbf{u}_w(\mathbf{x}_t,t)
\right]\Delta t
+
\sqrt{2K\Delta t}\mathbf{Z}
$$

where:

$$
\mathbf{Z}\sim\mathcal{N}(\mathbf{0},\mathbf{I})
$$

For backward source reconstruction:

$$
\mathbf{x}_{t-\Delta t}
=
\mathbf{x}_t
-
\left[
\mathbf{u}_c(\mathbf{x}_t,t)
+
\alpha\mathbf{u}_w(\mathbf{x}_t,t)
\right]\Delta t
+
\sqrt{2K\Delta t}\mathbf{Z}
$$

The ensemble distribution at earlier times represents the probable
source region of the observed spill.

In [ ]:
# =========================
# Stochastic SDE Parameters
# =========================

ALPHA = WINDAGE

K = DIFFUSION_COEFFICIENT

DT_HOURS = TIME_STEP_HOURS
DT_SECONDS = DT_HOURS * 3600.0

print("Windage coefficient α:", ALPHA)
print("Diffusion coefficient K:", K, "m²/s")
print("Time step:", DT_HOURS, "hours")
print("Time step:", DT_SECONDS, "seconds")

In [ ]:
def stochastic_backward_step(
    particle,
    environment,
    dt_seconds,
    windage,
    diffusion
):
    """
    Perform one stochastic backward Euler-Maruyama step.

    x(t-dt) =
        x(t)
        - [current + windage * wind] * dt
        + sqrt(2*K*dt) * Z

    Z ~ N(0, I)
    """

    # Deterministic transport velocity
    u = (
        environment.current_u_mps
        + windage * environment.wind_u_mps
    )

    v = (
        environment.current_v_mps
        + windage * environment.wind_v_mps
    )

    # Backward deterministic displacement
    dx_deterministic = -u * dt_seconds
    dy_deterministic = -v * dt_seconds

    # Stochastic diffusion term
    sigma = np.sqrt(
        2.0 * diffusion * dt_seconds
    )

    dx_random = np.random.normal(0.0, sigma)
    dy_random = np.random.normal(0.0, sigma)

    # Total displacement
    dx = dx_deterministic + dx_random
    dy = dy_deterministic + dy_random

    # Convert metres → geographic coordinates
    lat_delta = meters_to_latitude(dy)

    lon_delta = meters_to_longitude(
        dx,
        particle.latitude
    )

    return Particle(
        latitude=particle.latitude + lat_delta,
        longitude=particle.longitude + lon_delta
    )

In [ ]:
def simulate_one_backward_particle(
    observed_particle,
    lookback_hours,
    environment,
    windage,
    diffusion,
    dt_hours
):

    particle = observed_particle

    trajectory = [
        {
            "hours_before_observation": 0,
            "latitude": particle.latitude,
            "longitude": particle.longitude
        }
    ]

    num_steps = int(
        lookback_hours / dt_hours
    )

    for step in range(1, num_steps + 1):

        particle = stochastic_backward_step(
            particle,
            environment,
            dt_hours * 3600.0,
            windage,
            diffusion
        )

        trajectory.append({
            "hours_before_observation": step * dt_hours,
            "latitude": particle.latitude,
            "longitude": particle.longitude
        })

    return pd.DataFrame(trajectory)

In [ ]:
single_trajectory = simulate_one_backward_particle(
    particle,
    LOOKBACK_HOURS,
    environment,
    ALPHA,
    K,
    DT_HOURS
)

single_trajectory.head()

In [ ]:
plt.figure(figsize=(9, 7))

plt.plot(
    single_trajectory["longitude"],
    single_trajectory["latitude"],
    marker="o",
    markersize=3
)

plt.scatter(
    OBSERVED_LON,
    OBSERVED_LAT,
    s=120,
    label="Observed spill"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Single Stochastic Backward Trajectory")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
def generate_stochastic_ensemble(
    observed_particle,
    num_particles,
    lookback_hours,
    environment,
    windage,
    diffusion,
    dt_hours
):

    origins = []

    for particle_id in range(num_particles):

        particle = observed_particle

        num_steps = int(
            lookback_hours / dt_hours
        )

        for _ in range(num_steps):

            particle = stochastic_backward_step(
                particle,
                environment,
                dt_hours * 3600.0,
                windage,
                diffusion
            )

        origins.append({
            "particle_id": particle_id,
            "latitude": particle.latitude,
            "longitude": particle.longitude
        })

    return pd.DataFrame(origins)

In [ ]:
origins_df = generate_stochastic_ensemble(
    observed_particle=particle,
    num_particles=NUM_PARTICLES,
    lookback_hours=LOOKBACK_HOURS,
    environment=environment,
    windage=ALPHA,
    diffusion=K,
    dt_hours=DT_HOURS
)

print("Particles generated:", len(origins_df))
origins_df.head()

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    origins_df["longitude"],
    origins_df["latitude"],
    s=5,
    alpha=0.15,
    label="Backward particles"
)

plt.scatter(
    OBSERVED_LON,
    OBSERVED_LAT,
    s=120,
    marker="x",
    label="Observed spill"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(
    f"Stochastic Backward Ensemble "
    f"({NUM_PARTICLES} particles)"
)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
origin_latitude = origins_df["latitude"].mean()
origin_longitude = origins_df["longitude"].mean()

print("Estimated mean origin")
print("--------------------")
print(f"Latitude : {origin_latitude:.6f}")
print(f"Longitude: {origin_longitude:.6f}")

In [ ]:
origins_df["distance_from_mean_km"] = haversine_km(
    origin_latitude,
    origin_longitude,
    origins_df["latitude"],
    origins_df["longitude"]
)

origins_df["distance_from_mean_km"].describe()

In [ ]:
radius_95_km = origins_df[
    "distance_from_mean_km"
].quantile(0.95)

print(
    f"Estimated 95% origin radius: "
    f"{radius_95_km:.2f} km"
)

In [ ]:
def create_origin_probability_field(
    origins,
    bins=60
):

    histogram, lon_edges, lat_edges = np.histogram2d(
        origins["longitude"],
        origins["latitude"],
        bins=bins
    )

    total_particles = histogram.sum()

    probability = (
        histogram / total_particles
        if total_particles > 0
        else histogram
    )

    return probability, lon_edges, lat_edges

In [ ]:
probability, lon_edges, lat_edges = (
    create_origin_probability_field(
        origins_df,
        bins=60
    )
)

print("Probability sum:", probability.sum())

In [ ]:
max_index = np.unravel_index(
    np.argmax(probability),
    probability.shape
)

lon_index, lat_index = max_index

most_probable_lon = (
    lon_edges[lon_index]
    + lon_edges[lon_index + 1]
) / 2

most_probable_lat = (
    lat_edges[lat_index]
    + lat_edges[lat_index + 1]
) / 2

print("Highest-probability origin cell")
print("--------------------------------")
print(f"Latitude : {most_probable_lat:.6f}")
print(f"Longitude: {most_probable_lon:.6f}")

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(
    probability.T,
    origin="lower",
    extent=[
        lon_edges[0],
        lon_edges[-1],
        lat_edges[0],
        lat_edges[-1]
    ],
    aspect="auto"
)

plt.scatter(
    OBSERVED_LON,
    OBSERVED_LAT,
    marker="x",
    s=150,
    label="Observed spill"
)

plt.scatter(
    most_probable_lon,
    most_probable_lat,
    marker="+",
    s=180,
    label="Highest probability"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Probable Oil Spill Origin Field")

plt.colorbar(
    label="Origin Probability"
)

plt.legend()

plt.show()

In [ ]:
observed_at = datetime.now(timezone.utc)

origin_time_start = (
    observed_at
    - timedelta(hours=LOOKBACK_HOURS)
)

origin_time_end = observed_at

print("Observed time:")
print(observed_at.isoformat())

print("\nPossible origin window:")
print(origin_time_start.isoformat())
print("to")
print(origin_time_end.isoformat())

In [ ]:
origin_reconstruction = {
    "mode": "stochastic_hindcast",

    "observed_location": {
        "latitude": OBSERVED_LAT,
        "longitude": OBSERVED_LON
    },

    "estimated_origin": {
        "latitude": float(origin_latitude),
        "longitude": float(origin_longitude)
    },

    "highest_probability_origin": {
        "latitude": float(most_probable_lat),
        "longitude": float(most_probable_lon)
    },

    "uncertainty_radius_95_km": float(
        radius_95_km
    ),

    "time_window": {
        "start": origin_time_start.isoformat(),
        "end": origin_time_end.isoformat()
    },

    "particle_count": NUM_PARTICLES,

    "environment": {
        "current_u_mps": environment.current_u_mps,
        "current_v_mps": environment.current_v_mps,
        "wind_u_mps": environment.wind_u_mps,
        "wind_v_mps": environment.wind_v_mps,
        "windage": ALPHA,
        "diffusion_m2s": K
    }
}

origin_reconstruction

In [ ]:
OUTPUT_DIR = Path("../data/demo/drift")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

origins_df.to_csv(
    OUTPUT_DIR / "stochastic_origin_particles.csv",
    index=False
)

with open(
    OUTPUT_DIR / "origin_reconstruction.json",
    "w"
) as f:

    json.dump(
        origin_reconstruction,
        f,
        indent=2
    )

print(
    "Saved outputs to:",
    OUTPUT_DIR.resolve()
)

In [ ]:
print("=" * 50)
print("ORCA DRIFT EXPERIMENT SUMMARY")
print("=" * 50)

print(
    f"Observed location: "
    f"{OBSERVED_LAT:.4f}, {OBSERVED_LON:.4f}"
)

print(
    f"Particles: {NUM_PARTICLES}"
)

print(
    f"Lookback: {LOOKBACK_HOURS} hours"
)

print(
    f"Windage α: {ALPHA}"
)

print(
    f"Diffusion K: {K} m²/s"
)

print(
    f"Estimated origin: "
    f"{origin_latitude:.6f}, "
    f"{origin_longitude:.6f}"
)

print(
    f"95% uncertainty radius: "
    f"{radius_95_km:.2f} km"
)

print(
    f"Highest probability cell: "
    f"{most_probable_lat:.6f}, "
    f"{most_probable_lon:.6f}"
)

print(
    f"Origin window: "
    f"{origin_time_start.isoformat()} "
    f"→ {origin_time_end.isoformat()}"
)

print("=" * 50)